# Create AFNI 3dLMEr data table

## Functions

### Summarize data table

In [1]:
def summarize_dataframe(df, name="DataFrame"):
    """
    Generate comprehensive summary statistics for a dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to summarize
    name : str
        Name of the dataframe for display purposes
        
    Returns:
    --------
    tuple : (col_info, desc_stats)
        col_info: DataFrame with column-level information
        desc_stats: DataFrame with descriptive statistics
    """
    print(f"\n{'='*50}")
    print(f"Summary for: {name}")
    print(f"{'='*50}\n")
    
    # Show summary info
    df.info(verbose=True, show_counts=True)
    
    # Per-column summary table
    col_info = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.count(),
        'n_unique(incl_NaN)': df.nunique(dropna=False),
        'pct_missing': (df.isna().mean() * 100).round(2),
        'sample_values_first5': df.apply(lambda s: s.dropna().unique()[:5].tolist())
    })
    
    # Descriptive statistics
    desc_stats = df.describe(include='all').T
    
    return col_info, desc_stats

# Usage example:
# merged_col_info, merged_desc_all = summarize_dataframe(merged_df, "merged_df")

## Load relevant modules|

In [2]:
# Old Windows WSL paths (kept as reference)
# participants_input_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\Behavioral\data\participants_gb.tsv"
# data_table_format_input_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\fmri\data\LME_longitudinal_datatable.txt"
# output_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\fmri\results\3dlmerdata_table_with_covariates.txt"


# Define input file paths using local Linux paths
# participants_gb.tsv contains demographic and clinical data for all participants
participants_input_path = "~/Documents/GutBrain/data/participants_gb.tsv"

# 3dLMEr_table.txt is the template data table with fMRI file paths and basic structure
data_table_format_input_path = "~/Documents/GutBrain/BIDS_results/3dLmer/3dLMEr_table.txt"

# qc_final_decision_clean.csv contains quality control decisions for excluding bad scans
qc_decision_path = "~/Documents/GutBrain/derivatives7/qc_final_decision_clean.csv"

# Output path not set yet - will be defined later when saving results
# output_path = "~/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates.txt"

In [ ]:
import pandas as pd
import os

# Load QC decision file
qc_decision_path_expanded = os.path.expanduser(qc_decision_path)
qc_df = pd.read_csv(qc_decision_path_expanded)

# Show comprehensive summary
print("=" * 80)
print("QC DECISION FILE SUMMARY")
print("=" * 80)

# Basic info
print(f"\nTotal rows: {len(qc_df)}")
print(f"Total columns: {len(qc_df.columns)}")

# Show all column names
print("\n" + "=" * 80)
print("COLUMN NAMES:")
print("=" * 80)
print(qc_df.columns.tolist())

# Key QC columns summary
print("\n" + "=" * 80)
print("KEY QC DECISION COLUMNS:")
print("=" * 80)

# BIDS name breakdown
print(f"\nUnique BIDS names (sub-sess-run combos): {qc_df['bids_name'].nunique()}")
print(f"Sample BIDS names:\n{qc_df['bids_name'].head(10).tolist()}")

# Decision column
if 'Decision finale' in qc_df.columns:
    print(f"\nDecision finale value counts:")
    print(qc_df['Decision finale'].value_counts())
    
if 'decision' in qc_df.columns:
    print(f"\nDecision (intermediate) value counts:")
    print(qc_df['decision'].value_counts())

# Session breakdown
if 'Session' in qc_df.columns:
    print(f"\nSession distribution:")
    print(qc_df['Session'].value_counts().sort_index())

# Check for exclusion reasons
if 'Commentaire final' in qc_df.columns:
    print(f"\nExclusion comments (non-empty):")
    print(qc_df[qc_df['Commentaire final'].notna()]['Commentaire final'].value_counts())

# Descriptive statistics for QC metrics
print("\n" + "=" * 80)
print("QC METRICS SUMMARY:")
print("=" * 80)
qc_metrics = ['fd_mean', 'fd_max', 'fd_num', 'fd_num_above_3mm', 'dvars_nstd', 
              'tsnr', 'Coverage_percentage']
available_metrics = [m for m in qc_metrics if m in qc_df.columns]
print(qc_df[available_metrics].describe())

# Detailed column info
print("\n" + "=" * 80)
print("DETAILED COLUMN INFORMATION:")
print("=" * 80)
qc_col_info = pd.DataFrame({
    'dtype': qc_df.dtypes.astype(str),
    'non_null': qc_df.count(),
    'n_unique': qc_df.nunique(),
    'pct_missing': (qc_df.isna().mean() * 100).round(2),
    'sample_values': qc_df.apply(lambda s: s.dropna().unique()[:3].tolist())
})
print(qc_col_info)

# Show first few rows
print("\n" + "=" * 80)
print("FIRST 5 ROWS:")
print("=" * 80)
print(qc_df.head())

In [ ]:
import pandas as pd
import re
import os

# Define the conversion function first
def from_afni_id(x):
    """Convert 'sub-RCX096' → 'rcx096', 'sub-RND001' → 'rnd001'"""
    m = re.match(r"sub-([A-Z]+)(\d+)", x)
    if m:
        prefix = m.group(1).lower()
        number = m.group(2).zfill(3)
        return f"{prefix}{number}"
    return x

# Load QC decision file
qc_decision_path_expanded = os.path.expanduser(qc_decision_path)
qc_df = pd.read_csv(qc_decision_path_expanded)

# Parse bids_name to extract subject, session, and run
def parse_bids_name(bids_name):
    """
    Parse BIDS name like 'sub-RCX097_ses-1_task-BDM_run-1_bold'
    Returns dict with subject, session, run
    """
    # Extract subject (e.g., 'sub-RCX097' -> 'rcx097')
    subj_match = re.search(r'sub-([A-Z]+\d+)', bids_name)
    subj = subj_match.group(1).lower() if subj_match else None
    
    # Extract session (e.g., 'ses-1' -> 1)
    sess_match = re.search(r'ses-(\d+)', bids_name)
    sess = int(sess_match.group(1)) if sess_match else None
    
    # Extract run (e.g., 'run-1' -> 1)
    run_match = re.search(r'run-(\d+)', bids_name)
    run = int(run_match.group(1)) if run_match else None
    
    return {'Subj': subj, 'sess': sess, 'run': run}

# Parse all bids_name entries
qc_parsed = qc_df['bids_name'].apply(parse_bids_name).apply(pd.Series)
qc_df_with_parsed = pd.concat([qc_df, qc_parsed], axis=1)

# Clean up the decision column by stripping whitespace
if 'Decision finale' in qc_df_with_parsed.columns:
    qc_df_with_parsed['Decision finale'] = qc_df_with_parsed['Decision finale'].astype(str).str.strip()
    approved_scans = qc_df_with_parsed[qc_df_with_parsed['Decision finale'] == 'inclure'].copy()
    print(f"Using 'Decision finale' column with value 'inclure'")
elif 'decision' in qc_df_with_parsed.columns:
    qc_df_with_parsed['decision'] = qc_df_with_parsed['decision'].astype(str).str.strip()
    approved_scans = qc_df_with_parsed[qc_df_with_parsed['decision'] == 'include'].copy()
    print(f"Using 'decision' column with value 'include'")
else:
    print("ERROR: No decision column found!")
    approved_scans = qc_df_with_parsed.copy()

print(f"\nTotal scans in QC file: {len(qc_df_with_parsed)}")
print(f"Approved scans: {len(approved_scans)}")
print(f"Excluded scans: {len(qc_df_with_parsed) - len(approved_scans)}")

# Create base dataframe with only Subj, sess, run from approved QC scans
base_df = approved_scans[['Subj', 'sess', 'run']].copy()

# Load and prepare participants_df
print("Loading participants_df...")
participants_df = pd.read_csv(os.path.expanduser(participants_input_path), sep='\t')
participants_df = participants_df.rename(columns={'participant_id': 'Subj', 'session': 'sess'})
participants_df['Subj'] = participants_df['Subj'].astype(str).apply(from_afni_id)
participants_df['sess'] = pd.to_numeric(participants_df['sess'], errors='coerce')

# Fix handedness for rcx247 and rnd076
if 'handedness' in participants_df.columns:
    participants_df.loc[participants_df['Subj'] == 'rcx247', 'handedness'] = 'right'
    participants_df.loc[participants_df['Subj'] == 'rnd076', 'handedness'] = 'right'
    print("\nFixed handedness for rcx247 and rnd076 to 'right'")

# Merge with participants data
merged_df = pd.merge(
    base_df,
    participants_df,
    how='left',
    on=['Subj', 'sess'],
    suffixes=('', '_part')
)

# Drop unwanted columns (including TWL)
cols_to_drop = [col.strip() for col in [
    'weight', 'imc_recherche', 'ewl_recherche', 'twl_recherche',
    'bi_ttaille', 'bi_thanche', 'bi_tcou',
    'tx_t2d___3', 'qte_boost', 'tx_t2d___2', 'age', 'vas_fin', 't2d', ' tx_t2d___1',
    'twl_24months'  # Remove TWL@24months if it exists
]]
merged_df = merged_df.drop(columns=[col for col in cols_to_drop if col in merged_df.columns])

# Grand mean center selected columns
for col in ['vas_debut', 'age_baseline', 'bmi_baseline']:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col] - merged_df[col].mean()

# Convert sess, type_chx, and run to string for categorical treatment in AFNI
for col in ['sess', 'type_chx', 'run']:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].astype(int).astype(str)

# Impute missing values by mean within each session
numeric_cols = merged_df.select_dtypes(include='number').columns
participant_col = 'id_participant' if 'id_participant' in merged_df.columns else 'Subj'

for col in numeric_cols:
    session_means = merged_df.groupby('sess')[col].transform('mean')
    imputed_mask = merged_df[col].isna() & session_means.notna()

    if imputed_mask.any():
        for _, row in merged_df.loc[imputed_mask, [participant_col, 'sess']].iterrows():
            print(f"Imputed column='{col}' for {participant_col}='{row[participant_col]}', session='{row['sess']}'")

    merged_df[col] = merged_df[col].fillna(session_means)

print(f"\n{'='*80}")
print("FINAL MERGED DATAFRAME SUMMARY:")
print(f"{'='*80}")
print(f"Total rows: {len(merged_df)}")
print(f"Unique subjects: {merged_df['Subj'].nunique()}")
print(f"Unique sessions: {merged_df['sess'].nunique()}")
print(f"\nScans per session:")
print(merged_df['sess'].value_counts().sort_index())

# Check handedness
if 'handedness' in merged_df.columns:
    print(f"\nHandedness distribution:")
    print(merged_df[['Subj', 'handedness']].drop_duplicates().groupby('handedness').size())
    print(f"\nrcx247 handedness: {merged_df[merged_df['Subj'] == 'rcx247']['handedness'].unique()}")
    print(f"rnd076 handedness: {merged_df[merged_df['Subj'] == 'rnd076']['handedness'].unique()}")

# Display first few rows
merged_df.head()

Using 'Decision finale' column with value 'inclure'

Total scans in QC file: 507
Approved scans: 507
Excluded scans: 0
Loading participants_df...

Fixed handedness for rcx247 and rnd076 to 'right'

FINAL MERGED DATAFRAME SUMMARY:
Total rows: 507
Unique subjects: 57
Unique sessions: 4

Scans per session:
sess
1    146
2    138
3    112
4    111
Name: count, dtype: int64

Handedness distribution:
handedness
left      3
right    54
dtype: int64

rcx247 handedness: ['right']
rnd076 handedness: ['right']


,Subj,sess,run,sex,type_chx,handedness,vas_debut,age_baseline,bmi_baseline
0,rcx097,1,1,female,3,right,0.073987,-4.134714,0.107199
1,rcx097,1,2,female,3,right,0.073987,-4.134714,0.107199
2,rcx097,2,1,female,3,right,-3.162124,-4.134714,0.107199
3,rcx097,2,2,female,3,right,-3.162124,-4.134714,0.107199
4,rcx097,2,3,female,3,right,-3.162124,-4.134714,0.107199


In [ ]:
# Define output path for the data tables
import os

# Set the output directory and base filename
output_dir = os.path.expanduser("~/Documents/GutBrain/BIDS_results/3dLmer")
base_filename = "3dlmerdata_table_with_covariates_qc_final.txt"

# Create the full output path
output_path = os.path.join(output_dir, base_filename)

# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"Output path set to: {output_path}")

Output path set to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_qc_final.txt


In [ ]:
# --- Create data table for view_Hi_vs_view_low_v7_literature_complex_psc ---

def make_view_Hi_vs_view_low_v7_literature_complex_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_view_Hi_vs_view_low_v7_literature_complex_psc.nii.gz"

# Create a new table with the same rows but new InputFile paths
view_Hi_vs_view_low_v7_literature_complex_psc_table = merged_df.copy()
view_Hi_vs_view_low_v7_literature_complex_psc_table['InputFile'] = view_Hi_vs_view_low_v7_literature_complex_psc_table.apply(
    make_view_Hi_vs_view_low_v7_literature_complex_psc_inputfile, axis=1
)

# Save to a new file with qc_final suffix at the end
output_path_view_Hi_vs_view_low_v7_literature_complex_psc = output_path.replace(
    '_qc_final.txt', '_view_Hi_vs_view_low_v7_literature_complex_psc_qc_final.txt'
)
view_Hi_vs_view_low_v7_literature_complex_psc_table.to_csv(
    output_path_view_Hi_vs_view_low_v7_literature_complex_psc, sep='\t', index=False, lineterminator='\n'
)

print(f"New table with view_Hi_vs_view_low_v7_literature_complex_psc InputFile column saved to: {output_path_view_Hi_vs_view_low_v7_literature_complex_psc}")

# --- Create data table for mod_hi_vs_mod_lo_v7_literature_complex_psc ---

def make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_hi_vs_mod_lo_v7_literature_complex_psc.nii.gz"

# Create a new table with the same rows but new InputFile paths
mod_hi_vs_mod_lo_v7_literature_complex_psc_table = merged_df.copy()
mod_hi_vs_mod_lo_v7_literature_complex_psc_table['InputFile'] = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.apply(make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile, axis=1)

# Save to a new file with qc_final suffix at the end
output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc = output_path.replace('_qc_final.txt', '_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt')
mod_hi_vs_mod_lo_v7_literature_complex_psc_table.to_csv(output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc, sep='\t', index=False, lineterminator='\n')

print(f"New table with mod_hi_vs_mod_lo_v7_literature_complex_psc InputFile column saved to: {output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc}")

# --- Create data table for v7_literature_complex_mod_view_Lo_vs_0_psc ---

def make_v7_literature_complex_mod_view_Lo_vs_0_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_view_Lo_vs_0_psc.nii.gz"

# Create a new table with the same rows but new InputFile paths
v7_literature_complex_mod_view_Lo_vs_0_psc_table = merged_df.copy()
v7_literature_complex_mod_view_Lo_vs_0_psc_table['InputFile'] = v7_literature_complex_mod_view_Lo_vs_0_psc_table.apply(
    make_v7_literature_complex_mod_view_Lo_vs_0_psc_inputfile, axis=1
)

# Save to a new file with qc_final suffix at the end
output_path_v7_literature_complex_mod_view_Lo_vs_0_psc = output_path.replace(
    '_qc_final.txt', '_v7_literature_complex_mod_view_Lo_vs_0_psc_qc_final.txt'
)
v7_literature_complex_mod_view_Lo_vs_0_psc_table.to_csv(
    output_path_v7_literature_complex_mod_view_Lo_vs_0_psc, sep='\t', index=False, lineterminator='\n'
)

print(f"New table with v7_literature_complex_mod_view_Lo_vs_0_psc InputFile column saved to: {output_path_v7_literature_complex_mod_view_Lo_vs_0_psc}")

# --- Create data table for v7_literature_complex_mod_view_Hi_vs_0_psc ---

def make_v7_literature_complex_mod_view_Hi_vs_0_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_view_Hi_vs_0_psc.nii.gz"

# Create a new table with the same rows but new InputFile paths
v7_literature_complex_mod_view_Hi_vs_0_psc_table = merged_df.copy()
v7_literature_complex_mod_view_Hi_vs_0_psc_table['InputFile'] = v7_literature_complex_mod_view_Hi_vs_0_psc_table.apply(
    make_v7_literature_complex_mod_view_Hi_vs_0_psc_inputfile, axis=1
)

# Save to a new file with qc_final suffix at the end
output_path_v7_literature_complex_mod_view_Hi_vs_0_psc = output_path.replace(
    '_qc_final.txt', '_v7_literature_complex_mod_view_Hi_vs_0_psc_qc_final.txt'
)
v7_literature_complex_mod_view_Hi_vs_0_psc_table.to_csv(
    output_path_v7_literature_complex_mod_view_Hi_vs_0_psc, sep='\t', index=False, lineterminator='\n'
)

print(f"New table with v7_literature_complex_mod_view_Hi_vs_0_psc InputFile column saved to: {output_path_v7_literature_complex_mod_view_Hi_vs_0_psc}")

New table with view_Hi_vs_view_low_v7_literature_complex_psc InputFile column saved to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_view_Hi_vs_view_low_v7_literature_complex_psc_qc_final.txt
New table with mod_hi_vs_mod_lo_v7_literature_complex_psc InputFile column saved to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt
New table with v7_literature_complex_mod_view_Lo_vs_0_psc InputFile column saved to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_v7_literature_complex_mod_view_Lo_vs_0_psc_qc_final.txt
New table with v7_literature_complex_mod_view_Hi_vs_0_psc InputFile column saved to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_v7_literature_complex_mod_view_Hi_vs_0_psc_qc_final.txt


## diagnose missing rows in final data tables

In [ ]:
# --- Create data table for mod_hi_vs_mod_lo_v7_literature_complex_psc ---

def make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_hi_vs_mod_lo_v7_literature_complex_psc.nii.gz"

# Create a new table with the same rows but new InputFile paths
mod_hi_vs_mod_lo_v7_literature_complex_psc_table = merged_df.copy()
mod_hi_vs_mod_lo_v7_literature_complex_psc_table['InputFile'] = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.apply(make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile, axis=1)

# ========== DIAGNOSTIC CODE ==========
print("\n" + "="*80)
print("DIAGNOSTIC: mod_hi_vs_mod_lo_v7_literature_complex_psc_table")
print("="*80)

# Check shape
print(f"\nDataFrame shape: {mod_hi_vs_mod_lo_v7_literature_complex_psc_table.shape}")
print(f"Columns: {len(mod_hi_vs_mod_lo_v7_literature_complex_psc_table.columns)}")
print(f"Rows: {len(mod_hi_vs_mod_lo_v7_literature_complex_psc_table)}")

# Check for NaN values per column
nan_counts = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.isna().sum()
if nan_counts.any():
    print("\n⚠️  Columns with NaN values:")
    print(nan_counts[nan_counts > 0])
    
    # Show specific problematic rows (lines 87-94, 498-506 correspond to row indices 86-93, 497-505)
    problematic_indices = list(range(86, 94)) + list(range(497, 506))
    existing_indices = [i for i in problematic_indices if i < len(mod_hi_vs_mod_lo_v7_literature_complex_psc_table)]
    
    if existing_indices:
        print(f"\n⚠️  Checking rows that correspond to problematic lines (indices {existing_indices}):")
        problematic_rows = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.iloc[existing_indices]
        
        # Show which columns have NaN in these rows
        for idx in existing_indices:
            row_nan_cols = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.iloc[idx].isna()
            if row_nan_cols.any():
                print(f"\n  Row {idx} (line {idx+2} in file) has NaN in columns:")
                print(f"    {row_nan_cols[row_nan_cols].index.tolist()}")
                print(f"    Subj: {mod_hi_vs_mod_lo_v7_literature_complex_psc_table.iloc[idx]['Subj']}")
                print(f"    sess: {mod_hi_vs_mod_lo_v7_literature_complex_psc_table.iloc[idx]['sess']}")
                print(f"    run: {mod_hi_vs_mod_lo_v7_literature_complex_psc_table.iloc[idx]['run']}")
else:
    print("\n✓ No NaN values found")

# Check data types
print(f"\nData types:")
print(mod_hi_vs_mod_lo_v7_literature_complex_psc_table.dtypes)

# Fill NaN values before saving
print("\n" + "="*80)
print("FIXING NaN VALUES")
print("="*80)

for col in mod_hi_vs_mod_lo_v7_literature_complex_psc_table.columns:
    if mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col].isna().any():
        n_missing = mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col].isna().sum()
        if mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col].dtype in ['float64', 'int64']:
            # For numeric, fill with 0
            mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col] = mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col].fillna(0)
            print(f"  Filled {n_missing} NaN values in '{col}' with 0")
        else:
            # For categorical/string, fill with empty string
            mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col] = mod_hi_vs_mod_lo_v7_literature_complex_psc_table[col].fillna('')
            print(f"  Filled {n_missing} NaN values in '{col}' with empty string")

# Verify no NaN left
remaining_nan = mod_hi_vs_mod_lo_v7_literature_complex_psc_table.isna().sum().sum()
print(f"\nRemaining NaN values after filling: {remaining_nan}")

print("\n" + "="*80)
# ========== END DIAGNOSTIC CODE ==========

# Save to a new file with qc_final suffix at the end
output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc = output_path.replace('_qc_final.txt', '_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt')
mod_hi_vs_mod_lo_v7_literature_complex_psc_table.to_csv(
    output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc, 
    sep='\t', 
    index=False, 
    lineterminator='\n',
    na_rep=''  # Ensure any remaining NaN are written as empty string
)

print(f"New table with mod_hi_vs_mod_lo_v7_literature_complex_psc InputFile column saved to: {output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc}")

# Verify the saved file
print("\n" + "="*80)
print("VERIFYING SAVED FILE")
print("="*80)
with open(output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc, 'r') as f:
    lines = f.readlines()
    header = lines[0].strip().split('\t')
    n_cols = len(header)
    print(f"Expected columns: {n_cols}")
    
    # Check problematic lines
    for line_num in [87, 88, 89, 90, 91, 92, 93, 94, 498, 499, 500, 501, 502, 503, 504, 505, 506]:
        if line_num < len(lines):
            cols = lines[line_num].strip().split('\t')
            if len(cols) != n_cols:
                print(f"  ⚠️  Line {line_num+1}: Expected {n_cols} cols, got {len(cols)} (missing {n_cols - len(cols)})")
            else:
                print(f"  ✓ Line {line_num+1}: OK ({len(cols)} columns)")


DIAGNOSTIC: mod_hi_vs_mod_lo_v7_literature_complex_psc_table

DataFrame shape: (507, 10)
Columns: 10
Rows: 507

✓ No NaN values found

Data types:
Subj             object
sess             object
run              object
sex              object
type_chx         object
handedness       object
vas_debut       float64
age_baseline    float64
bmi_baseline    float64
InputFile        object
dtype: object

FIXING NaN VALUES

Remaining NaN values after filling: 0

New table with mod_hi_vs_mod_lo_v7_literature_complex_psc InputFile column saved to: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt

VERIFYING SAVED FILE
Expected columns: 10
  ✓ Line 88: OK (10 columns)
  ✓ Line 89: OK (10 columns)
  ✓ Line 90: OK (10 columns)
  ✓ Line 91: OK (10 columns)
  ✓ Line 92: OK (10 columns)
  ✓ Line 93: OK (10 columns)
  ✓ Line 94: OK (10 columns)
  ✓ Line 95: OK (10 columns)
  ✓ Line 499: OK (10 columns)
  ✓ Li

## Tables for TWL model

In [ ]:
# ============================================================
# CREATE TWL_RECH_24 VARIABLE AND ADD TO MERGED_DF
# ============================================================

print("="*80)
print("CREATING TWL_RECH_24 VARIABLE")
print("="*80)

# Extract TWL from session 4 and map to all sessions of the same participant
twl_24_mapping = participants_df[participants_df['sess'] == 4][['Subj', 'twl_recherche']].copy()
twl_24_mapping = twl_24_mapping.rename(columns={'twl_recherche': 'twl_rech_24'})
twl_24_mapping = twl_24_mapping.dropna(subset=['twl_rech_24'])

print(f"\nSubjects with TWL@24months data: {len(twl_24_mapping)}")
print(f"Sample values:\n{twl_24_mapping.head()}")

# Merge twl_rech_24 into merged_df
merged_df = pd.merge(
    merged_df,
    twl_24_mapping,
    on='Subj',
    how='left'
)

print(f"\ntwl_rech_24 added to merged_df")
print(f"Available for {merged_df['twl_rech_24'].notna().sum()} out of {len(merged_df)} rows")
print(f"TWL range: {merged_df['twl_rech_24'].min():.3f} to {merged_df['twl_rech_24'].max():.3f}")

# ============================================================
# FILTER DATA FOR TWL MODEL (twl_rech_24 <= 0.6)
# ============================================================

print("\n" + "="*80)
print("FILTERING DATA FOR TWL MODEL")
print("="*80)

rows_before_filter = len(merged_df)
merged_df_twl = merged_df[merged_df['twl_rech_24'] <= 0.6].copy()
rows_after_filter = len(merged_df_twl)

print(f"\nFiltered out {rows_before_filter - rows_after_filter} rows where twl_rech_24 > 0.6")
print(f"Rows before: {rows_before_filter}, Rows after: {rows_after_filter}")
print(f"Unique subjects before: {merged_df['Subj'].nunique()}")
print(f"Unique subjects after: {merged_df_twl['Subj'].nunique()}")

print(f"\nSample size per session after filtering:")
print(merged_df_twl.groupby('sess')['Subj'].nunique())

# ============================================================
# CREATE DATA TABLES FOR TWL MODEL
# ============================================================

print("\n" + "="*80)
print("CREATING DATA TABLES FOR TWL MODEL")
print("="*80)

# --- Create data table for mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24 ---

def make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_hi_vs_mod_lo_v7_literature_complex_psc.nii.gz"

mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table = merged_df_twl.copy()
mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table['InputFile'] = mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table.apply(
    make_mod_hi_vs_mod_lo_v7_literature_complex_psc_inputfile, axis=1
)

# Move InputFile to the end
cols = [col for col in mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table.columns if col != 'InputFile'] + ['InputFile']
mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table = mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table[cols]

output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24 = output_path.replace(
    '_qc_final.txt', '_mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_qc_final.txt'
)
mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_table.to_csv(
    output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24, 
    sep='\t', 
    index=False, 
    lineterminator='\n'
)

print(f"✓ mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24 saved to:")
print(f"  {output_path_mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24}")

# --- Create data table for mod_view_vs_0_v7_literature_psc_twl_24 ---

def make_mod_view_vs_0_v7_literature_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_mod_view_vs_0_v7_literature_psc.nii.gz"

mod_view_vs_0_v7_literature_psc_twl_24_table = merged_df_twl.copy()
mod_view_vs_0_v7_literature_psc_twl_24_table['InputFile'] = mod_view_vs_0_v7_literature_psc_twl_24_table.apply(
    make_mod_view_vs_0_v7_literature_psc_inputfile, axis=1
)

# Move InputFile to the end
cols = [col for col in mod_view_vs_0_v7_literature_psc_twl_24_table.columns if col != 'InputFile'] + ['InputFile']
mod_view_vs_0_v7_literature_psc_twl_24_table = mod_view_vs_0_v7_literature_psc_twl_24_table[cols]

output_path_mod_view_vs_0_v7_literature_psc_twl_24 = output_path.replace(
    '_qc_final.txt', '_mod_view_vs_0_v7_literature_psc_twl_24_qc_final.txt'
)
mod_view_vs_0_v7_literature_psc_twl_24_table.to_csv(
    output_path_mod_view_vs_0_v7_literature_psc_twl_24, 
    sep='\t', 
    index=False, 
    lineterminator='\n'
)

print(f"✓ mod_view_vs_0_v7_literature_psc_twl_24 saved to:")
print(f"  {output_path_mod_view_vs_0_v7_literature_psc_twl_24}")

# --- Create data table for view_Hi_vs_view_low_v7_literature_complex_psc_twl_24 ---

def make_view_Hi_vs_view_low_v7_literature_complex_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_view_Hi_vs_view_low_v7_literature_complex_psc.nii.gz"

view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table = merged_df_twl.copy()
view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table['InputFile'] = view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table.apply(
    make_view_Hi_vs_view_low_v7_literature_complex_psc_inputfile, axis=1
)

# Move InputFile to the end
cols = [col for col in view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table.columns if col != 'InputFile'] + ['InputFile']
view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table = view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table[cols]

output_path_view_Hi_vs_view_low_v7_literature_complex_psc_twl_24 = output_path.replace(
    '_qc_final.txt', '_view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_qc_final.txt'
)
view_Hi_vs_view_low_v7_literature_complex_psc_twl_24_table.to_csv(
    output_path_view_Hi_vs_view_low_v7_literature_complex_psc_twl_24, 
    sep='\t', 
    index=False, 
    lineterminator='\n'
)

print(f"✓ view_Hi_vs_view_low_v7_literature_complex_psc_twl_24 saved to:")
print(f"  {output_path_view_Hi_vs_view_low_v7_literature_complex_psc_twl_24}")

# --- Create data table for v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24 ---

def make_v7_literature_complex_mod_view_Hi_vs_0_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_view_Hi_vs_0_psc.nii.gz"

v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table = merged_df_twl.copy()
v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table['InputFile'] = v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table.apply(
    make_v7_literature_complex_mod_view_Hi_vs_0_psc_inputfile, axis=1
)

# Move InputFile to the end
cols = [col for col in v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table.columns if col != 'InputFile'] + ['InputFile']
v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table = v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table[cols]

output_path_v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24 = output_path.replace(
    '_qc_final.txt', '_v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_qc_final.txt'
)
v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24_table.to_csv(
    output_path_v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24, 
    sep='\t', 
    index=False, 
    lineterminator='\n'
)

print(f"✓ v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24 saved to:")
print(f"  {output_path_v7_literature_complex_mod_view_Hi_vs_0_psc_twl_24}")

# --- Create data table for v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24 ---

def make_v7_literature_complex_mod_view_Lo_vs_0_psc_inputfile(row):
    subj = str(row['Subj']).upper()
    sess = str(row['sess'])
    run = str(row['run'])
    return f"/home/pagag24/projects/def-amichaud/share/GutBrain/BIDS_results2/sub-{subj}/ses-{sess}/func/sub-{subj}_ses-{sess}_run-{run}_effect-size_v7_literature_complex_mod_view_Lo_vs_0_psc.nii.gz"

v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table = merged_df_twl.copy()
v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table['InputFile'] = v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table.apply(
    make_v7_literature_complex_mod_view_Lo_vs_0_psc_inputfile, axis=1
)

# Move InputFile to the end
cols = [col for col in v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table.columns if col != 'InputFile'] + ['InputFile']
v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table = v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table[cols]

output_path_v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24 = output_path.replace(
    '_qc_final.txt', '_v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_qc_final.txt'
)
v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24_table.to_csv(
    output_path_v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24, 
    sep='\t', 
    index=False, 
    lineterminator='\n'
)

print(f"✓ v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24 saved to:")
print(f"  {output_path_v7_literature_complex_mod_view_Lo_vs_0_psc_twl_24}")

# Summary
print("\n" + "="*80)
print("SUMMARY: TWL MODEL DATA TABLES")
print("="*80)
print(f"Total tables created: 5")
print(f"All tables include 'twl_rech_24' column")
print(f"All tables filtered to: twl_rech_24 <= 0.6")
print(f"Total rows in each table: {len(merged_df_twl)}")
print(f"Unique subjects: {merged_df_twl['Subj'].nunique()}")
print(f"\nAll output filenames have '_twl_24_qc_final.txt' suffix")
print(f"InputFile paths point to existing first-level PSC effect files")

CREATING TWL_RECH_24 VARIABLE

Subjects with TWL@24months data: 72
Sample values:
      Subj  twl_rech_24
3   rcx096     0.285106
7   rnd001     0.406602
11  rnd002     0.428571
19  rnd003     0.370479
23  rcx097     0.358097

twl_rech_24 added to merged_df
Available for 485 out of 507 rows
TWL range: 0.168 to 0.992

FILTERING DATA FOR TWL MODEL

Filtered out 36 rows where twl_rech_24 > 0.6
Rows before: 507, Rows after: 471
Unique subjects before: 57
Unique subjects after: 51

Sample size per session after filtering:
sess
1    51
2    47
3    43
4    44
Name: Subj, dtype: int64

CREATING DATA TABLES FOR TWL MODEL
✓ mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24 saved to:
  /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_twl_24_qc_final.txt
✓ mod_view_vs_0_v7_literature_psc_twl_24 saved to:
  /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/3dlmerdata_table_with_covariates_mod_view_vs_0_v7_literatu